# MICE補完の可視化と検証
このノートブックでは、 で行われているMICE補完が適切に動作しているかを確認します。
特に、補完された値が前後のデータや他の限月（金利カーブ）と整合性が取れているかを可視化します。

In [ ]:
import sys
import os
# プロジェクトルートをパスに追加
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.processing import load_and_clean_data

# データの読み込み
df = load_and_clean_data("../data/BOJ_data.xlsx", "../data/BOJ_meeting_history.csv")
print(f"Total rows: {len(df)}")
df.head()

## 1. 補完状況の統計
各カラムでどれくらいの欠損があり、補完されたかを確認します。

In [ ]:
# M1-M8だけでなく、外部指標の補完フラグも作成されているか確認
# 注: src/processing.py では M1-M8, T12-T24 に対して _is_imputed フラグを作成している
imputed_cols = [c for c in df.columns if "_is_imputed" in c]
stats = df[imputed_cols].sum().to_frame(name="imputed_count")
stats["imputed_rate(%)"] = (stats["imputed_count"] / len(df) * 100).round(2)
print("--- 補完統計 ---")
display(stats.sort_values("imputed_count", ascending=False))

# 外部指標（USDJPY等）に欠損が残っていないか確認
ext_cols = ["USDJPY", "JGB_Future", "Nikkei225", "DXY"]
print("
--- 外部指標の残存欠損数（MICE後は0になるはず） ---")
print(df[ext_cols].isnull().sum())

## 2. 時系列での補完箇所の可視化
補完された値を赤点でプロットし、前後の実データと馴染んでいるかを確認します。

In [ ]:
def plot_imputation(df, col, start_date=None, end_date=None):
    plot_df = df.copy()
    if start_date:
        plot_df = plot_df[plot_df["Date"] >= start_date]
    if end_date:
        plot_df = plot_df[plot_df["Date"] <= end_date]
    
    plt.figure(figsize=(15, 5))
    # 実データ部分
    actual = plot_df[plot_df[f"{col}_is_imputed"] == 0]
    plt.plot(plot_df["Date"], plot_df[col], color="gray", alpha=0.5, label="Imputed Line")
    plt.scatter(actual["Date"], actual[col], color="blue", s=10, label="Actual Data")
    
    # 補完データ部分
    imputed = plot_df[plot_df[f"{col}_is_imputed"] == 1]
    plt.scatter(imputed["Date"], imputed[col], color="red", s=30, marker="x", label="Imputed Data")
    
    plt.title(f"Imputation Visualization: {col}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# 欠損が多かったカラム（例: M1）を直近1年間で表示
latest_date = df["Date"].max()
one_year_ago = latest_date - pd.Timedelta(days=365)
plot_imputation(df, "M1", start_date=one_year_ago)

## 3. 金利カーブ（M1-M8）の整合性確認
補完が行われた特定の「日」において、M1〜M8のカーブ形状が不自然になっていないかを確認します。

In [ ]:
# 補完が含まれる日をいくつかピックアップ
imputed_days = df[df[imputed_cols].sum(axis=1) > 0].sample(5, random_state=42)

m_cols = [f"M{i}" for i in range(1, 9)]

plt.figure(figsize=(12, 6))
for _, row in imputed_days.iterrows():
    label = row["Date"].strftime("%Y-%m-%d")
    plt.plot(range(1, 9), row[m_cols], marker="o", label=label)
    
    # 補完されている点は×印で強調
    for i, col in enumerate(m_cols):
        if row[f"{col}_is_imputed"] == 1:
            plt.scatter(i+1, row[col], color="red", s=100, marker="x", zorder=5)

plt.title("Yield Curve (M1-M8) on Imputed Days (Red X = Imputed)")
plt.xlabel("Meeting Index")
plt.ylabel("Rate (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. 外部指標との相関の維持
MICEが外部指標（USDJPY等）との関係を壊していないか散布図で確認します。

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="USDJPY", y="M8", hue="M8_is_imputed", palette={0: "blue", 1: "red"}, alpha=0.5)
plt.title("Correlation Check: USDJPY vs M8 (Red = Imputed M8)")
plt.show()

## 5. 外部指標（USDJPY等）の補完確認
外部指標自体もMICEで補完されています。USDJPYなどの動きが不自然になっていないか確認します。

In [ ]:
# USDJPYの補完状況を確認（MICE前の元データに欠損があった箇所を特定するために、
# 今回は再読み込みして欠損箇所を特定します）
raw_data = pd.read_excel("../data/BOJ_data.xlsx").iloc[1:]
raw_data["日付"] = pd.to_datetime(raw_data["日付"], format="%Y年%m月%d日")
raw_usdjpy_null_dates = raw_data[raw_data["JPY= (MID_PRICE)"].isnull()]["日付"]

plt.figure(figsize=(15, 5))
plt.plot(df["Date"], df["USDJPY"], color="blue", alpha=0.5, label="USDJPY (MICE Imputed)")

# 元々欠損していた日をハイライト
imputed_usdjpy = df[df["Date"].isin(raw_usdjpy_null_dates)]
plt.scatter(imputed_usdjpy["Date"], imputed_usdjpy["USDJPY"], color="red", marker="x", label="Original was Missing")

plt.title("USDJPY Imputation Check (Red X = Values estimated by MICE)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(one_year_ago, latest_date)
plt.show()